# N2 · 构建一份 rebuttal (Rebuttal Builder)

> 配套 9.7-L3 · **真实科研动作**: 对一组 (样例) 审稿意见做分类 + 优先级 + 字数预算,
> 生成结构化 rebuttal 骨架, 再亲手写一段「补实验」的有证据回应。

论文的命运常在 rebuttal 阶段决定。这是把"被审判"变成"系统解题"的训练。

In [1]:
import sys
from pathlib import Path
SRC = Path.cwd().parent / "src"
sys.path.insert(0, str(SRC))
import rebuttal_kit as rk
print("意见分类:", list(rk.CATEGORIES.keys()))

意见分类: ['factual_error', 'add_experiment', 'clarification', 'weakness', 'scope', 'positive']


## 1. 一组真实感的审稿意见 (3 个 reviewer 的混合)

含: 要补实验、说不清楚、正面、范围之争、真实弱点 —— 典型的一轮审稿。

In [2]:
comments = [
    "The baseline DPO appears under-tuned; please compare against a properly tuned DPO.",
    "It is unclear how the robust weighting in Eq.3 is computed. The notation is confusing.",
    "This is an interesting and well-motivated paper with a clear contribution.",
    "Only 7B is tested. Why not show it scales to 70B? Otherwise feels limited in scope.",
    "Robustness is only studied under one type of label noise, which is a real concern.",
    "Missing an ablation on the weighting hyperparameter.",
]
print(f"{len(comments)} 条意见待处理")

6 条意见待处理


## 2. 分类 + 优先级 (别情绪化, 翻译成「他要什么」)

In [3]:
triaged = rk.triage(comments)
print(f"{'优先级':<6}{'类别':<16}意见")
print("-"*70)
for t in triaged:
    print(f"P{t['priority']:<5}{t['category']:<16}{t['comment'][:46]}...")
print("\n→ '补实验'类排最前 (能直接翻盘); '正面评价'垫底 (不耗字数)。")

优先级   类别              意见
----------------------------------------------------------------------
P3    add_experiment  The baseline DPO appears under-tuned; please c...
P3    add_experiment  Missing an ablation on the weighting hyperpara...
P2    clarification   It is unclear how the robust weighting in Eq.3...
P2    clarification   Only 7B is tested. Why not show it scales to 7...
P2    weakness        Robustness is only studied under one type of l...
P0    positive        This is an interesting and well-motivated pape...

→ '补实验'类排最前 (能直接翻盘); '正面评价'垫底 (不耗字数)。


## 3. 按优先级分配有限字数 (rebuttal 常有硬字数上限)

In [4]:
budgeted = rk.budget_words(triaged, total_words=500)
for t in budgeted:
    print(f"  [{t['category']:<16}] ≈{t['word_budget']:>3} 词  ← {t['comment'][:40]}...")
print("\n→ 火力集中在能翻盘的点, 而非平均用力。")

  [add_experiment  ] ≈123 词  ← The baseline DPO appears under-tuned; pl...
  [add_experiment  ] ≈123 词  ← Missing an ablation on the weighting hyp...
  [clarification   ] ≈ 82 词  ← It is unclear how the robust weighting i...
  [clarification   ] ≈ 82 词  ← Only 7B is tested. Why not show it scale...
  [weakness        ] ≈ 82 词  ← Robustness is only studied under one typ...
  [positive        ] ≈  8 词  ← This is an interesting and well-motivate...

→ 火力集中在能翻盘的点, 而非平均用力。


## 4. 生成 rebuttal 骨架 (往里填)

In [5]:
skel = rk.build_skeleton(comments, total_words=500)
out = Path.cwd() / "_paper_output"; out.mkdir(exist_ok=True)
(out / "rebuttal_skeleton.md").write_text(skel, encoding="utf-8")
print(skel[:900])

# Rebuttal 骨架 (auto)

> 原则: 先谢评审; 按对录用影响排序优先回应; 能补实验是 rebuttal 期最强武器;
> 礼貌、就事论事、对每条都回应(哪怕一句)。绝不情绪化、不无视、不空泛。

感谢各位审稿人的细致意见。下面逐条回应 (R=Reviewer)。

## [1] [补充实验] (≈123 词)
> 意见: The baseline DPO appears under-tuned; please compare against a properly tuned DPO.
> 策略: 能补就补(rebuttal 期补实验最有力); 不能补则说明计划
> 回应: ____________

## [2] [补充实验] (≈123 词)
> 意见: Missing an ablation on the weighting hyperparameter.
> 策略: 能补就补(rebuttal 期补实验最有力); 不能补则说明计划
> 回应: ____________

## [3] [澄清说明] (≈82 词)
> 意见: It is unclear how the robust weighting in Eq.3 is computed. The notation is confusing.
> 策略: 解释 + 承诺正文改写, 引用具体段落
> 回应: ____________

## [4] [澄清说明] (≈82 词)
> 意见: Only 7B is tested. Why not show it scales to 70B? Otherwise feels limited in scope.
> 策略: 解释 + 承诺正文改写, 引用具体段落
> 回应: ____________

## [5] [承认并缓解] (≈82 词)
> 意见: Robustness is only studied under one type of label noise, which is a real concern.
> 策略: 承认 + 缓解


## 5. 亲手写一段: 回应「补实验」(最强武器)

骨架给了结构, 真正的回应要你写。下面示范「补实验」类回应的范式: **用证据, 不辩解**。

In [6]:
draft = """[补充实验] R1-Q1 (baseline 调参):
We thank R1 for this point. We re-ran DPO with the SAME hyperparameter search
budget as Robust-DPO (20 trials of random search over lr and beta). The tuned
DPO improves from 0.40 to 0.43 at noise=0.4, but Robust-DPO still leads by +10
points (0.53), and the gap remains significant (p<0.001, 8 seeds). We have added
the tuned-DPO results to Table 2 and the search protocol to Appendix B."""
print(draft)
print("\n范式拆解:")
print("  ✓ 先谢 (礼貌)  ✓ 给同等调参预算 (回应公平性, 9.4-L3)")
print("  ✓ 给具体数字 (用证据)  ✓ 承诺已加进正文/附录 (可验证)")
print("  ✗ 没有一句'我们认为我们的方法很好'式的空泛辩解")

[补充实验] R1-Q1 (baseline 调参):
We thank R1 for this point. We re-ran DPO with the SAME hyperparameter search
budget as Robust-DPO (20 trials of random search over lr and beta). The tuned
DPO improves from 0.40 to 0.43 at noise=0.4, but Robust-DPO still leads by +10
points (0.53), and the gap remains significant (p<0.001, 8 seeds). We have added
the tuned-DPO results to Table 2 and the search protocol to Appendix B.

范式拆解:
  ✓ 先谢 (礼貌)  ✓ 给同等调参预算 (回应公平性, 9.4-L3)
  ✓ 给具体数字 (用证据)  ✓ 承诺已加进正文/附录 (可验证)
  ✗ 没有一句'我们认为我们的方法很好'式的空泛辩解


## 6. 反思 (9.7 收口)

你刚把一堆审稿意见从「情绪冲击」变成了「排好序的解题清单」, 并写了一段有证据的回应。带走:
- 收到意见先冷静, **系统分类** → 翻译成「他要什么」。
- 按**对录用影响**分配有限字数; **补实验是最强翻盘武器** (你 9.4 的快速实验能力变现)。
- 语气礼貌、用证据不辩解、逐条回应、归错于"表述不清"给台阶。
- rebuttal 也写给 AC 看; 全程保持专业 (风度是加分项)。

> **9.7 全线回顾**: 装配(L1) → 投稿(L2) → 评审+rebuttal(L3) → 录用/拒稿(L4)。
> **第一篇论文的目标是跑完整个循环**, 不是中顶会。

> 交棒 9.8: 论文录用 (或没录) 你都要把工作**讲给人听** ——
> 下一专题 `research-presentation` 教你 talk / poster / 答辩 / 电梯演讲。